# 02 — Join signal → risk → execution

Joins `signal_snapshots`, `risk_decisions`, and `fills` on `correlation_id`.

**Invariants**: V2 (every emitted signal has a risk decision joinable by correlation_id).

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src import db

In [ ]:
# Load signal snapshots (sum_arb: both prices present)
sig_rows = db.query("""
    SELECT
        correlation_id,
        contract_id,
        direction,
        poly_yes_price,
        poly_no_price,
        spread,
        expected_edge,
        position_size_usd,
        decision,
        rejection_reason,
        snapshot_time
    FROM signal_snapshots
    WHERE poly_yes_price IS NOT NULL AND poly_no_price IS NOT NULL
""")
signals = pd.DataFrame(sig_rows)
print(f'Signals: {len(signals)}')

In [ ]:
# Load risk decisions
risk_rows = db.query("""
    SELECT
        correlation_id,
        risk_level,
        rejection_reason  AS risk_rejection_reason,
        gate_latency_us,
        checked_at
    FROM risk_decisions
    WHERE correlation_id IS NOT NULL
""")
risk = pd.DataFrame(risk_rows)
print(f'Risk decisions: {len(risk)}')

In [ ]:
# Load fills
fill_rows = db.query("""
    SELECT
        correlation_id,
        fill_price,
        fill_size,
        remaining_size,
        submit_latency_us,
        fill_latency_us,
        venue_status,
        reject_reason     AS fill_reject_reason,
        submit_time,
        fill_time
    FROM fills
    WHERE correlation_id IS NOT NULL
""")
fills = pd.DataFrame(fill_rows)
print(f'Fills: {len(fills)}')

In [ ]:
# Join: signal LEFT JOIN risk LEFT JOIN fills
joined = (
    signals
    .merge(risk,  on='correlation_id', how='left')
    .merge(fills, on='correlation_id', how='left')
)
print(f'Joined rows: {len(joined)}')
joined.head(3)

In [ ]:
# V2: every emitted signal must have a risk decision
emitted = joined[joined['decision'] == 'emitted']
missing_risk = emitted['risk_level'].isna()
print(f'Emitted signals: {len(emitted)}')
print(f'V2 violations (emitted with no risk decision): {missing_risk.sum()}')
if missing_risk.sum() > 0:
    print('WARN: V2 — some emitted signals have no risk record:')
    print(emitted[missing_risk][['correlation_id','snapshot_time']].to_string())

In [ ]:
# Save joined frame for downstream notebooks
joined.to_parquet('../data/raw/joined.parquet', index=False)
print('Saved data/raw/joined.parquet')